# Grammar scoring from spoken audio

Audio answers are transcribed with Whisper, scored by fine-tuned transformer regressors, and combined by a stacker. The metric is mean absolute error (MAE) against a 1.0 to 5.0 grammar score.

## Workflow

1. **Transcribe.** Whisper converts each audio file to text. A disfluency-preserving prompt keeps fillers, repeats and false starts, which carry grammar signal. Per-segment confidence and pause statistics are saved alongside the text.
2. **Build features.** Text statistics (word count, sentence length, type-token ratio, filler and repeat counts) join the Whisper statistics and the audio duration to form a small table of fluency features.
3. **Fine-tune.** Four transformer regressors (DeBERTa-v3 large and base, RoBERTa large and base) are trained with 5-fold stratified cross-validation. Each produces out-of-fold predictions on the full training set and a fold-averaged prediction on the test set.
4. **Stack.** Single models, a plain mean, an MAE-optimal blend, a Ridge stack and two gradient-boosted stackers compete on repeated-CV MAE. The winner is refit on all training rows.
5. **Submit.** Predictions are clipped to the label range, snapped to the 0.5 label grid when that lowers MAE, and written to `submission.csv`.

## Design choices

| Choice | Why |
|---|---|
| 5-fold stratified CV, not a single split | A 110-row holdout swung MAE from 0.43 to 0.77 between epochs, so single-split results were not usable for model selection |
| Best epoch selected on MAE, not Pearson | The metric is MAE; a high-correlation epoch can be badly biased |
| SmoothL1 loss on standardised labels | Closer to MAE than MSE, and standardising removes the wasted first epochs spent rescaling |
| Cosine LR decay to zero over the epochs actually run | Late epochs fine-tune at a small learning rate, which steadies validation MAE |
| Mean pooling with a linear head, dropout off | A stronger regression head than CLS pooling on this amount of data |
| Layer-wise LR decay plus a separate head LR | Protects the pretrained layers while the fresh head trains faster |
| Dynamic padding up to 512 tokens, gradient accumulation to batch 16 | No truncation of long answers, and steadier gradients than a batch of 4 |
| Disfluency-preserving Whisper prompt | Whisper repairs repeats and fillers by default, erasing fluency evidence |
| Stack over fold-averaged models | Lower variance than any single model, and the stacker corrects residual bias from the fluency features |

Fold predictions are cached under `RUN_DIR`, so an interrupted session resumes fold by fold. Changing a training setting requires deleting that model's folder in `RUN_DIR`, otherwise the cached predictions are reused.

Out-of-fold MAE is slightly optimistic, because each fold's best epoch is chosen on that fold's own validation rows. Compare variants under the same protocol and treat the leaderboard as the final judge.


## 0. Environment

Pinned to `transformers==4.40.0`, the version the DeBERTa-v3 and RoBERTa configurations below were tuned against.


In [1]:
!pip install -q openai-whisper tqdm transformers==4.40.0 scipy scikit-learn sentencepiece protobuf soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


## 1. Imports, settings and paths

Every knob for the run lives in this cell. `RUN_DIR` holds the transcript cache, per-fold predictions and all submission files.


In [ ]:
import gc
import json
import math
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import whisper
from scipy.optimize import minimize
from scipy.stats import pearsonr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import RepeatedKFold, StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore', category=FutureWarning)

# ----- experiment settings -----
SEED = 42
N_FOLDS = 5
N_SEEDS = 1
EPOCHS = 10             
PATIENCE = 4             # stop a fold after this many epochs without a validation MAE gain
WARMUP_FRAC = 0.1
MAX_TOKEN_LEN = 512      
WEIGHT_DECAY = 0.01
LOSS_BETA = 0.3          
DISABLE_DROPOUT = True
USE_AMP = True           

WHISPER_MODEL = 'small'
USE_DISFLUENCY_PROMPT = True
DISFLUENCY_PROMPT = "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. I I was going to, uh, to say, yeah."

RUN_MODELS = ['deberta_v3_large', 'roberta_large', 'deberta_v3_base', 'roberta_base']
MODEL_CONFIGS = {
    'deberta_v3_large': dict(checkpoint='microsoft/deberta-v3-large', batch_size=4, accum=4, lr=2e-5, llrd=0.95, head_lr=2e-4),
    'roberta_large':    dict(checkpoint='roberta-large',              batch_size=8, accum=2, lr=1.5e-5, llrd=0.95, head_lr=2e-4),
    'deberta_v3_base':  dict(checkpoint='microsoft/deberta-v3-base',  batch_size=16, accum=1, lr=3e-5, llrd=0.90, head_lr=5e-4),
    'roberta_base':     dict(checkpoint='roberta-base',               batch_size=16, accum=1, lr=3e-5, llrd=0.90, head_lr=5e-4),
}

# ----- paths -----
DATASET_NAME = 'shl-hiring-assessment-2026'
BASE_DIR = Path(f'/kaggle/input/competitions/{DATASET_NAME}/Dataset_Final')
TRAIN_CSV = BASE_DIR / 'train.csv'
TEST_CSV = BASE_DIR / 'test.csv'
TRAIN_AUDIO = BASE_DIR / 'train'
TEST_AUDIO = BASE_DIR / 'test'

RUN_DIR = Path('/kaggle/working/model_comparison_v2')
TAG = 'disfl' if USE_DISFLUENCY_PROMPT else 'plain'
TRANSCRIPT_DIR = RUN_DIR / 'transcripts'
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_TRANSCRIPTS = TRANSCRIPT_DIR / f'train_transcripts_{TAG}.csv'
TEST_TRANSCRIPTS = TRANSCRIPT_DIR / f'test_transcripts_{TAG}.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ON = USE_AMP and DEVICE.type == 'cuda'


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f'Device: {DEVICE} | AMP: {AMP_ON} | folds: {N_FOLDS} | epochs/fold: {EPOCHS} | models: {RUN_MODELS}')

Device: cuda | AMP: True | folds: 5 | epochs/fold: 10 | models: ['deberta_v3_large', 'roberta_large', 'deberta_v3_base', 'roberta_base']


## 2. Transcription and fluency features

Whisper normally cleans speech: it drops fillers, collapses stutters and repairs false starts. For grammar scoring those are the evidence. The initial prompt biases Whisper towards keeping them. Per-segment confidence and pause statistics, plus audio duration, are cached with the transcript for later use by the stacker.


In [3]:
WHISPER_COLS = ['seg_logprob_mean', 'seg_logprob_min', 'seg_no_speech_max', 'seg_compression_mean',
                'n_segments', 'pause_total', 'pause_max', 'n_long_pauses', 'speech_span']
CACHE_COLS = ['filename', 'transcript', 'duration'] + WHISPER_COLS

_whisper_model = None


def get_whisper():
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model(WHISPER_MODEL)
    return _whisper_model


def audio_duration(path):
    try:
        import soundfile as sf
        return float(sf.info(str(path)).duration)
    except Exception:
        return len(whisper.load_audio(str(path))) / 16000.0


def segment_features(segments):
    if not segments:
        return {c: np.nan for c in WHISPER_COLS}
    lens = np.array([max(s['end'] - s['start'], 1e-3) for s in segments])
    logprob = np.array([s['avg_logprob'] for s in segments])
    gaps = np.array([max(segments[i + 1]['start'] - segments[i]['end'], 0.0) for i in range(len(segments) - 1)]
                    if len(segments) > 1 else [0.0])
    return {
        'seg_logprob_mean': float(np.average(logprob, weights=lens)),
        'seg_logprob_min': float(logprob.min()),
        'seg_no_speech_max': float(max(s['no_speech_prob'] for s in segments)),
        'seg_compression_mean': float(np.mean([s['compression_ratio'] for s in segments])),
        'n_segments': len(segments),
        'pause_total': float(gaps.sum()),
        'pause_max': float(gaps.max()),
        'n_long_pauses': int((gaps > 0.5).sum()),
        'speech_span': float(segments[-1]['end'] - segments[0]['start']),
    }


def transcribe_files(csv_path, audio_dir, output_path):
    rows = pd.read_csv(csv_path)
    records = {}
    if output_path.exists():
        previous = pd.read_csv(output_path)
        if all(c in previous.columns for c in CACHE_COLS):
            previous['transcript'] = previous['transcript'].fillna('').astype(str)
            previous = previous[previous['transcript'].str.strip() != '']
            records = previous.set_index('filename')[CACHE_COLS[1:]].to_dict('index')

    pending = rows[~rows['filename'].isin(records)].reset_index(drop=True)
    print(f'{output_path.name}: {len(records)} cached, {len(pending)} to transcribe')

    def save(final=False):
        df = pd.DataFrame([{'filename': k, **v} for k, v in records.items()], columns=CACHE_COLS)
        if final:
            df = rows.merge(df, on='filename', how='left')
        df.to_csv(output_path, index=False)

    for i, row in enumerate(tqdm(pending.itertuples(index=False), total=len(pending), desc=f'Transcribing {output_path.stem}'), start=1):
        path = audio_dir / row.filename
        try:
            kwargs = dict(language='en', fp16=(DEVICE.type == 'cuda'), condition_on_previous_text=False, temperature=0)
            if USE_DISFLUENCY_PROMPT:
                kwargs['initial_prompt'] = DISFLUENCY_PROMPT
            result = get_whisper().transcribe(str(path), **kwargs)
            records[row.filename] = {'transcript': result['text'].strip(), 'duration': audio_duration(path),
                                     **segment_features(result.get('segments', []))}
        except Exception as error:
            print(f'Could not transcribe {row.filename}: {error}')
            records[row.filename] = {'transcript': '', 'duration': np.nan, **{c: np.nan for c in WHISPER_COLS}}
        if i % 10 == 0:
            save()
    save(final=True)
    return pd.read_csv(output_path)


train_transcripts = transcribe_files(TRAIN_CSV, TRAIN_AUDIO, TRAIN_TRANSCRIPTS)
test_transcripts = transcribe_files(TEST_CSV, TEST_AUDIO, TEST_TRANSCRIPTS)

global_whisper = _whisper_model
_whisper_model = None
del global_whisper
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

for df in (train_transcripts, test_transcripts):
    df['transcript'] = df['transcript'].fillna('').astype(str)
print(f'Train rows: {len(train_transcripts)} (empty: {(train_transcripts.transcript.str.strip() == "").sum()}) | '
      f'Test rows: {len(test_transcripts)} (empty: {(test_transcripts.transcript.str.strip() == "").sum()})')

train_transcripts_disfl.csv: 0 cached, 769 to transcribe


Transcribing train_transcripts_disfl:   0%|          | 0/769 [00:00<?, ?it/s]


  0%|                                               | 0.00/461M [00:00<?, ?iB/s]
  0%|                                       | 128k/461M [00:00<06:18, 1.28MiB/s]
  0%|                                       | 864k/461M [00:00<01:38, 4.91MiB/s]
  1%|▍                                     | 5.29M/461M [00:00<00:20, 23.8MiB/s]
  3%|█▎                                    | 16.0M/461M [00:00<00:07, 58.5MiB/s]
  6%|██▏                                   | 26.1M/461M [00:00<00:06, 75.6MiB/s]
  7%|██▊                                   | 34.4M/461M [00:00<00:05, 79.2MiB/s]
  9%|███▍                                  | 41.9M/461M [00:00<00:05, 74.1MiB/s]
 11%|████                                  | 49.1M/461M [00:00<00:05, 72.7MiB/s]
 12%|████▋                                 | 56.2M/461M [00:00<00:05, 73.2MiB/s]
 14%|█████▏                                | 63.2M/461M [00:01<00:06, 68.6MiB/s]
 15%|█████▊                                | 69.8M/461M [00:01<00:06, 62.2MiB/s]
 17%|██████▍               

test_transcripts_disfl.csv: 0 cached, 216 to transcribe


Transcribing test_transcripts_disfl:   0%|          | 0/216 [00:00<?, ?it/s]

Train rows: 769 (empty: 0) | Test rows: 216 (empty: 0)


## 3. Cleaning, text features and cross-validation folds

Transcripts are whitespace-normalised, rows without a usable label or transcript are dropped, and text statistics are joined to the Whisper statistics. Folds are stratified on the rounded label so every fold sees the full score range.


In [4]:
FILLERS = re.compile(r"\b(?:um+|uh+|er+|erm|ah+|hmm+|mm+)\b", re.I)
REPEATS = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)


def clean_text(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def text_features(text):
    words = re.findall(r"[A-Za-z']+", text)
    lower = [w.lower() for w in words]
    n = len(words)
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    return {
        'n_words': n,
        'n_sents': len(sentences),
        'avg_sent_len': n / max(len(sentences), 1),
        'avg_word_len': float(np.mean([len(w) for w in words])) if n else 0.0,
        'ttr': len(set(lower)) / n if n else 0.0,
        'n_fillers': len(FILLERS.findall(text)),
        'n_repeats': len(REPEATS.findall(text)),
        'n_like': lower.count('like'),
    }


def build_features(df):
    tf = pd.DataFrame([text_features(t) for t in df['transcript']])
    out = pd.concat([tf, df[['duration'] + WHISPER_COLS].reset_index(drop=True)], axis=1)
    out['wpm'] = out['n_words'] / out['duration'].clip(lower=1.0) * 60
    out['filler_rate'] = out['n_fillers'] / out['n_words'].clip(lower=1)
    out['repeat_rate'] = out['n_repeats'] / out['n_words'].clip(lower=1)
    out['pause_ratio'] = out['pause_total'] / out['speech_span'].clip(lower=1.0)
    return out


train_rows = train_transcripts.copy()
train_rows['label'] = pd.to_numeric(train_rows['label'], errors='coerce')
train_rows['transcript'] = train_rows['transcript'].map(clean_text)
test_rows = test_transcripts.copy()
test_rows['transcript'] = test_rows['transcript'].map(clean_text)

keep = train_rows['transcript'].ne('') & train_rows['label'].notna() & (train_rows['label'] > 0)
train_rows = train_rows[keep].reset_index(drop=True)

feat_train = build_features(train_rows)
feat_test = build_features(test_rows)
FEATURE_COLS = list(feat_train.columns)

y = train_rows['label'].values.astype(float)
LABEL_LO, LABEL_HI = float(y.min()), float(y.max())
print(f'Training rows: {len(train_rows)} | label range [{LABEL_LO}, {LABEL_HI}] | mean {y.mean():.3f} | std {y.std():.3f}')
print('Label distribution:\n', train_rows['label'].round(1).value_counts().sort_index().to_string())


def make_bins(labels, n_splits):
    bins = np.clip(np.floor(labels), 0, 5).astype(int)
    counts = pd.Series(bins).value_counts().to_dict()
    while min(counts.values()) < n_splits and len(counts) > 1:
        smallest = min(counts, key=counts.get)
        nearest = min((b for b in counts if b != smallest), key=lambda b: abs(b - smallest))
        bins[bins == smallest] = nearest
        counts = pd.Series(bins).value_counts().to_dict()
    return bins


bins = make_bins(y, N_FOLDS)
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train_rows, bins))
print('Fold sizes (train/val):', [(len(a), len(b)) for a, b in folds])

Training rows: 732 | label range [1.0, 5.0] | mean 3.481 | std 1.014
Label distribution:
 label
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133
Fold sizes (train/val): [(585, 147), (585, 147), (586, 146), (586, 146), (586, 146)]


## 4. Model, dataset and training loop

A transformer encoder with masked mean pooling and a linear head. Training uses SmoothL1 on standardised labels, layer-wise learning-rate decay, cosine decay with warmup, mixed precision, and early stopping on validation MAE. The best epoch's weights are restored before predicting.


In [5]:
class TextDS(Dataset):
    def __init__(self, texts, targets, tokenizer):
        self.enc = tokenizer(list(texts), max_length=MAX_TOKEN_LEN, truncation=True)
        self.targets = [float(t) for t in targets]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'target': self.targets[i]}


def make_collate(tokenizer):
    def collate(batch):
        padded = tokenizer.pad({'input_ids': [b['input_ids'] for b in batch],
                                'attention_mask': [b['attention_mask'] for b in batch]}, return_tensors='pt')
        return {'input_ids': padded['input_ids'],
                'attention_mask': padded['attention_mask'],
                'target': torch.tensor([b['target'] for b in batch], dtype=torch.float32)}
    return collate


class Regressor(nn.Module):
    """Encoder + masked mean pooling + linear head (dropout optionally disabled)."""

    def __init__(self, checkpoint):
        super().__init__()
        config = AutoConfig.from_pretrained(checkpoint)
        if DISABLE_DROPOUT:
            config.hidden_dropout_prob = 0.0
            config.attention_probs_dropout_prob = 0.0
        self.backbone = AutoModel.from_pretrained(checkpoint, config=config)
        self.head = nn.Linear(config.hidden_size, 1)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state.float()
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return self.head(pooled).squeeze(-1)


def make_param_groups(model, lr, llrd, head_lr, weight_decay):
    """Layer-wise LR decay: top layer gets `lr`, each layer below gets `llrd` times the one above."""
    n_layers = model.backbone.config.num_hidden_layers
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith('head.'):
            group_lr = head_lr
        else:
            if name.startswith('backbone.embeddings'):
                depth = 0
            else:
                match = re.search(r'encoder\.layer\.(\d+)\.', name)
                depth = int(match.group(1)) + 1 if match else n_layers
            group_lr = lr * (llrd ** (n_layers - depth))
        no_decay = name.endswith('.bias') or 'LayerNorm' in name or 'layer_norm' in name
        groups.setdefault((group_lr, 0.0 if no_decay else weight_decay), []).append(param)
    return [{'params': ps, 'lr': l, 'weight_decay': w} for (l, w), ps in groups.items()]


@torch.no_grad()
def predict(model, loader):
    model.eval()
    outputs = []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ON):
            out = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        outputs.append(out.float().cpu().numpy())
    return np.concatenate(outputs)


def score(actual, predicted):
    predicted = np.asarray(predicted, dtype=float)
    pearson = pearsonr(actual, predicted)[0] if np.std(predicted) > 0 else 0.0
    return {'mae': float(mean_absolute_error(actual, predicted)),
            'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
            'pearson': float(pearson)}


def train_fold(cfg, tokenizer, tr_idx, va_idx, seed, tag):
    set_seed(seed)
    collate = make_collate(tokenizer)
    tr_y, va_y = y[tr_idx], y[va_idx]
    mu, sd = tr_y.mean(), tr_y.std()

    train_ds = TextDS(train_rows['transcript'].values[tr_idx], (tr_y - mu) / sd, tokenizer)
    valid_ds = TextDS(train_rows['transcript'].values[va_idx], np.zeros(len(va_idx)), tokenizer)
    test_ds = TextDS(test_rows['transcript'].values, np.zeros(len(test_rows)), tokenizer)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, collate_fn=collate)
    valid_loader = DataLoader(valid_ds, batch_size=16, shuffle=False, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate)

    model = Regressor(cfg['checkpoint']).to(DEVICE)
    optimizer = torch.optim.AdamW(
        make_param_groups(model, cfg['lr'], cfg['llrd'], cfg['head_lr'], WEIGHT_DECAY), lr=cfg['lr'], eps=1e-6)
    steps_per_epoch = math.ceil(len(train_loader) / cfg['accum'])
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=AMP_ON)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ON)
    loss_fn = nn.SmoothL1Loss(beta=LOSS_BETA)

    best_mae, best_epoch, best_state, bad_epochs = float('inf'), 0, None, 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{tag} ep{epoch}/{EPOCHS}', leave=False), start=1):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ON):
                prediction = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            loss = loss_fn(prediction.float(), batch['target'].to(DEVICE))
            scaler.scale(loss / cfg['accum']).backward()
            if step % cfg['accum'] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            running += loss.item()

        val_pred = np.clip(np.nan_to_num(predict(model, valid_loader) * sd + mu, nan=mu), LABEL_LO, LABEL_HI)
        metrics = score(va_y, val_pred)
        history.append({'epoch': epoch, 'train_loss': running / len(train_loader), **metrics})
        print(f'{tag} | ep {epoch:>2} | loss={running / len(train_loader):.4f} | MAE={metrics["mae"]:.4f} | '
              f'RMSE={metrics["rmse"]:.4f} | r={metrics["pearson"]:.4f}')

        if metrics['mae'] < best_mae or best_state is None:
            best_mae, best_epoch, bad_epochs = metrics['mae'], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f'{tag} | early stop (no MAE gain for {PATIENCE} epochs)')
                break

    model.load_state_dict(best_state)
    val_pred = np.clip(np.nan_to_num(predict(model, valid_loader) * sd + mu, nan=mu), LABEL_LO, LABEL_HI)
    test_pred = np.clip(np.nan_to_num(predict(model, test_loader) * sd + mu, nan=mu), LABEL_LO, LABEL_HI)

    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return val_pred, test_pred, history, best_epoch


def run_model(name):
    cfg = MODEL_CONFIGS[name]
    model_dir = RUN_DIR / name
    model_dir.mkdir(parents=True, exist_ok=True)
    final_path = model_dir / 'cv_predictions.npz'
    if final_path.exists():
        data = np.load(final_path)
        print(f'{name}: loaded cached CV predictions')
        return data['oof'], data['test']

    tokenizer = AutoTokenizer.from_pretrained(cfg['checkpoint'])
    oof = np.zeros((N_SEEDS, len(train_rows)))
    test_preds = []
    for s in range(N_SEEDS):
        for fold, (tr_idx, va_idx) in enumerate(folds):
            fold_path = model_dir / f'seed{s}_fold{fold}.npz'
            if fold_path.exists():
                data = np.load(fold_path)
                val_pred, test_pred = data['val'], data['test']
                print(f'{name} seed{s} fold{fold}: cached | MAE={mean_absolute_error(y[va_idx], val_pred):.4f}')
            else:
                tag = f'{name} s{s} f{fold}'
                val_pred, test_pred, history, best_epoch = train_fold(cfg, tokenizer, tr_idx, va_idx, SEED + 1000 * s + fold, tag)
                np.savez(fold_path, val=val_pred, test=test_pred)
                json.dump(history, open(model_dir / f'seed{s}_fold{fold}_history.json', 'w'))
                print(f'==> {tag}: best epoch {best_epoch} | MAE={mean_absolute_error(y[va_idx], val_pred):.4f}')
            oof[s, va_idx] = val_pred
            test_preds.append(test_pred)
    oof_mean, test_mean = oof.mean(0), np.mean(test_preds, axis=0)
    np.savez(final_path, oof=oof_mean, test=test_mean)
    return oof_mean, test_mean

## 5. Cross-validated training

Each model is trained once per fold. Out-of-fold predictions give an honest MAE across all training rows, and the fold models' test predictions are averaged.


In [6]:
oof_preds, test_preds, summary = {}, {}, []
for name in RUN_MODELS:
    print(f'\n===== {name} ({MODEL_CONFIGS[name]["checkpoint"]}) =====')
    oof_preds[name], test_preds[name] = run_model(name)
    metrics = score(y, oof_preds[name])
    fold_maes = [mean_absolute_error(y[va], oof_preds[name][va]) for _, va in folds]
    summary.append({'model': name, **{f'oof_{k}': v for k, v in metrics.items()}, 'fold_mae_std': float(np.std(fold_maes))})
    pd.DataFrame(summary).to_csv(RUN_DIR / 'model_comparison.csv', index=False)

oof_df = pd.DataFrame(oof_preds)
test_df = pd.DataFrame(test_preds)
display(pd.DataFrame(summary).sort_values('oof_mae'))


===== deberta_v3_large (microsoft/deberta-v3-large) =====


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

roberta_base s0 f4 ep9/10:   0%|          | 0/37 [00:00<?, ?it/s]

roberta_base s0 f4 | ep  9 | loss=0.0212 | MAE=0.4833 | RMSE=0.5960 | r=0.8060


roberta_base s0 f4 ep10/10:   0%|          | 0/37 [00:00<?, ?it/s]

roberta_base s0 f4 | ep 10 | loss=0.0162 | MAE=0.4791 | RMSE=0.5954 | r=0.8069
==> roberta_base s0 f4: best epoch 10 | MAE=0.4791


,model,oof_mae,oof_rmse,oof_pearson,fold_mae_std
0,deberta_v3_large,0.448857,0.571664,0.826081,0.024279
2,deberta_v3_base,0.459085,0.581128,0.819775,0.027763
1,roberta_large,0.468003,0.588577,0.814276,0.012580
3,roberta_base,0.468792,0.600746,0.806939,0.024921


## 6. Ensembling and stacking

Candidates are the single models, a plain mean, an MAE-optimal non-negative blend, a Ridge stack, and gradient-boosted stackers that also see the fluency features. Repeated 5-fold CV picks the winner by MAE. The features-only model is kept in the report as a baseline but excluded from selection.


In [7]:
class MeanModel:
    def fit(self, X, y_):
        return self

    def predict(self, X):
        return np.asarray(X).mean(axis=1)


class ColsModel:
    def __init__(self, estimator, cols):
        self.estimator, self.cols = estimator, cols

    def fit(self, X, y_):
        self.estimator.fit(X[self.cols].values, y_)
        return self

    def predict(self, X):
        return self.estimator.predict(X[self.cols].values)


class MAEBlend:
    """Non-negative weights summing to 1 that minimise MAE."""

    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y_):
        P = X[self.cols].values
        k = P.shape[1]
        result = minimize(lambda w: np.mean(np.abs(P @ w - y_)), np.ones(k) / k, method='SLSQP',
                          bounds=[(0, 1)] * k, constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}])
        self.w = result.x
        return self

    def predict(self, X):
        return X[self.cols].values @ self.w


def make_gbm(max_depth, lr, iters, leaf, l2):
    return HistGradientBoostingRegressor(loss='absolute_error', max_depth=max_depth, learning_rate=lr,
                                         max_iter=iters, min_samples_leaf=leaf, l2_regularization=l2,
                                         random_state=SEED)


class ResidualGBM:
    """Mean of models + a small, heavily regularised GBM correcting the residual from features."""

    def __init__(self, base_cols, feat_cols):
        self.base_cols, self.feat_cols = base_cols, feat_cols

    def fit(self, X, y_):
        base = X[self.base_cols].values.mean(axis=1)
        self.gbm = make_gbm(2, 0.03, 120, 25, 2.0).fit(X[self.feat_cols].values, y_ - base)
        return self

    def predict(self, X):
        return X[self.base_cols].values.mean(axis=1) + self.gbm.predict(X[self.feat_cols].values)


def cv_eval(factory, X, y_, n_repeats=3):
    rkf = RepeatedKFold(n_splits=5, n_repeats=n_repeats, random_state=SEED)
    preds = np.zeros((n_repeats, len(y_)))
    for i, (tr, va) in enumerate(rkf.split(X)):
        model = factory().fit(X.iloc[tr], y_[tr])
        preds[i // 5, va] = model.predict(X.iloc[va])
    maes = [mean_absolute_error(y_, np.clip(p, LABEL_LO, LABEL_HI)) for p in preds]
    return float(np.mean(maes)), np.clip(preds.mean(0), LABEL_LO, LABEL_HI)


pred_cols = list(oof_df.columns)
X_all = pd.concat([oof_df, feat_train], axis=1)
X_test_all = pd.concat([test_df, feat_test], axis=1)

candidates = {f'single:{c}': (lambda c=c: ColsModel(MeanModel(), [c])) for c in pred_cols}
candidates['mean_of_models'] = lambda: ColsModel(MeanModel(), pred_cols)
candidates['mae_weighted_blend'] = lambda: MAEBlend(pred_cols)
candidates['ridge_stack'] = lambda: ColsModel(Ridge(alpha=1.0), pred_cols)
candidates['gbm_stack_with_features'] = lambda: ColsModel(make_gbm(3, 0.05, 150, 20, 1.0), pred_cols + FEATURE_COLS)
candidates['residual_gbm'] = lambda: ResidualGBM(pred_cols, pred_cols + FEATURE_COLS)
candidates['features_only_gbm'] = lambda: ColsModel(make_gbm(3, 0.05, 150, 20, 1.0), FEATURE_COLS)

report, cv_oof = [], {}
for name, factory in candidates.items():
    mae, oof_cv = cv_eval(factory, X_all, y)
    cv_oof[name] = oof_cv
    report.append({'candidate': name, 'cv_mae': mae})
report = pd.DataFrame(report).sort_values('cv_mae').reset_index(drop=True)
report.to_csv(RUN_DIR / 'ensemble_report.csv', index=False)
display(report)

ensemble_candidates = [c for c in report['candidate'] if c != 'features_only_gbm']
best_name = ensemble_candidates[0]
print(f'Best candidate: {best_name} (CV MAE {report.set_index("candidate").loc[best_name, "cv_mae"]:.4f})')

,candidate,cv_mae
0,residual_gbm,0.428492
1,mean_of_models,0.442382
2,ridge_stack,0.443432
3,gbm_stack_with_features,0.443924
4,mae_weighted_blend,0.445364
5,single:deberta_v3_large,0.448857
6,single:deberta_v3_base,0.459085
7,single:roberta_large,0.468003
8,single:roberta_base,0.468792
9,features_only_gbm,0.593215


Best candidate: residual_gbm (CV MAE 0.4285)


## 7. Post-processing and submission

The winning stacker is refit on all training rows. Predictions are clipped to the observed label range, snapped to the 0.5 grid when that lowers OOF MAE, and written as `submission.csv`. One single-model submission per encoder is written alongside it as a fallback.


In [8]:
# Snap predictions to the label grid when the labels are a small discrete set and snapping lowers OOF MAE.
grid = np.unique(np.round(y, 4))


def snap(values):
    idx = np.abs(values[:, None] - grid[None, :]).argmin(axis=1)
    return grid[idx]


use_snap = False
if len(grid) <= 12:
    raw_mae = mean_absolute_error(y, cv_oof[best_name])
    snap_mae = mean_absolute_error(y, snap(cv_oof[best_name]))
    use_snap = snap_mae < raw_mae - 0.002
    print(f'Label grid {grid.tolist()} | MAE raw={raw_mae:.4f} snapped={snap_mae:.4f} -> snap={use_snap}')
else:
    print(f'{len(grid)} distinct labels: continuous-like, snapping skipped')

final_model = candidates[best_name]().fit(X_all, y)
final_test = np.clip(final_model.predict(X_test_all), LABEL_LO, LABEL_HI)
if use_snap:
    final_test = snap(final_test)

submission = pd.DataFrame({'filename': test_rows['filename'].values, 'label': final_test})
submission.to_csv(RUN_DIR / 'submission.csv', index=False)
submission.to_csv('/kaggle/working/submission.csv', index=False)

for name in pred_cols:
    pd.DataFrame({'filename': test_rows['filename'].values, 'label': test_df[name].values}).to_csv(
        RUN_DIR / f'submission_{name}.csv', index=False)

print(f'Saved final submission ({best_name}) with {len(submission)} rows')
print(submission['label'].describe().round(3).to_string())
display(submission.head())

Label grid [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0] | MAE raw=0.4273 snapped=0.4092 -> snap=True
Saved final submission (residual_gbm) with 216 rows
count    216.000
mean       3.275
std        0.864
min        2.000
25%        2.500
50%        3.000
75%        4.000
max        5.000


,filename,label
0,audio_128.wav,3.0
1,audio_156.wav,4.0
2,audio_19.wav,4.5
3,audio_108.wav,2.0
4,audio_206.wav,2.0


## Next steps

1. **Transcripts.** Run once with `USE_DISFLUENCY_PROMPT = True` and once with `False`, then compare the OOF MAE table. Try `WHISPER_MODEL = 'medium'`.
2. **More seeds.** `N_SEEDS = 3`, cached per fold, so it resumes after an interruption.
3. **Hyper-parameters.** `LOSS_BETA` from 0.1 to 0.5, per-model `lr`, `DISABLE_DROPOUT`, `EPOCHS` from 8 to 15.
4. **More diverse encoders.** Add `microsoft/deberta-v3-small` or `electra-large-discriminator` to `MODEL_CONFIGS`. Diversity helps a blend more than another seed does.
5. **Pseudo-labelling.** Add confident test predictions as extra training rows, once the CV setup is stable.
